In [1]:
from openai import OpenAI
import json

In [8]:
client = OpenAI(api_key="your_own_api_key")

In [9]:
MAX_ITERATIONS = 5
MAX_WORDS = 120

In [10]:
def generate_summary(text, feedback=None):
    system_prompt = (
        "You are a summarization agent. "
        "Produce a clear, beginner-friendly summary under 120 words. "
        "Avoid technical jargon. "
        "Apply feedback if provided."
    )
    
    user_prompt = f"TEXT:\n{text}\n"
    
    if feedback:
        user_prompt += f"\nFEEDBACK:\n{feedback}"
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    
    return response.choices[0].message.content.strip()

In [11]:
def evaluate_summary(summary):
    word_count = len(summary.split())
    
    system_prompt = (
        "You are a strict evaluator. "
        "Evaluate the summary against these criteria:\n"
        "- Beginner-friendly language\n"
        "- Clear and concise\n"
        "- No technical jargon\n\n"
        "Do NOT rewrite the summary.\n"
        "Return ONLY valid JSON in this format:\n"
        "{\n"
        '  "beginner_friendly": true/false,\n'
        '  "clarity": true/false,\n'
        '  "jargon_free": true/false,\n'
        '  "issues": [string],\n'
        '  "suggestions": [string]\n'
        "}"
    )
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": summary}
        ]
    )
    
    eval_data = json.loads(response.choices[0].message.content)
    
    eval_data["word_count_ok"] = word_count <= MAX_WORDS
    eval_data["word_count"] = word_count
    
    eval_data["passed"] = (
        eval_data["word_count_ok"]
        and eval_data["beginner_friendly"]
        and eval_data["clarity"]
        and eval_data["jargon_free"]
    )
    
    return eval_data

In [12]:
agent_memory = []

In [13]:
def run_agent(text):
    feedback = None
    best_summary = None
    best_score = -1

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n=== Iteration {iteration} ===")

        summary = generate_summary(text, feedback)
        evaluation = evaluate_summary(summary)

        # Compute a simple quality score
        score = sum([
            evaluation["word_count_ok"],
            evaluation["beginner_friendly"],
            evaluation["clarity"],
            evaluation["jargon_free"]
        ])

        # Save to memory
        agent_memory.append({
            "iteration": iteration,
            "summary": summary,
            "evaluation": evaluation,
            "feedback": feedback,
            "score": score
        })

        # Logging
        print(f"Word count: {evaluation['word_count']}")
        print(f"Checks passed: {score}/4")

        if evaluation["passed"]:
            print(" Goal achieved. Stopping.")
            return summary

        # Track best attempt
        if score > best_score:
            best_score = score
            best_summary = summary

        # Prepare feedback
        feedback = "\n".join(evaluation["suggestions"])
        print(" Failed checks:")
        for issue in evaluation["issues"]:
            print("-", issue)

        print(" Refining based on feedback...")

    print(" Max iterations reached. Returning best attempt.")
    return best_summary

In [14]:
input_text = """
AI agents are systems that use artificial intelligence to perform tasks
autonomously by perceiving their environment, making decisions, and taking
actions to achieve specific goals. They are often powered by large language
models and can use tools, memory, and planning strategies.
"""

final_summary = run_agent(input_text)
print("\nFINAL SUMMARY:\n", final_summary)


=== Iteration 1 ===
Word count: 43
Checks passed: 3/4
 Failed checks:
- The term 'sophisticated language tools' may not be clear to all beginners.
 Refining based on feedback...

=== Iteration 2 ===
Word count: 49
Checks passed: 3/4
 Failed checks:
- The term 'artificial intelligence' and 'language models' may be considered jargon for beginners.
 Refining based on feedback...

=== Iteration 3 ===
Word count: 55
Checks passed: 4/4
 Goal achieved. Stopping.

FINAL SUMMARY:
 AI agents are systems that use computer smartness to do tasks on their own. They watch their surroundings, make choices, and take actions to reach certain goals. These agents often rely on advanced chat systems, which help them understand and respond to language. They can also use tools, remember information, and plan their next steps.
